# ch 2. huggingface tokenizers

이번 챕터에서는 huggingface에서 제공하는 tokenizers 라이브러리를 이용해서 직접 corpus 데이터 셋을 가지고 subword tokenizer를 학습시켜 보겠습니다.

## 데이터 셋 준비

### 네이버 영화 리뷰 데이터 셋

네이버 영화 리뷰 데이터 셋으로 간단히 학습을 진행해보겠습니다.

In [2]:
import pandas as pd

In [3]:
train_df = pd.read_csv("./data/translation_train.csv")

In [4]:
train_df

,kor,eng
0,"mslee가 부재중이셔서, 당신에게 연락드립니다.","since ms.lee is absence, i contacted you."
1,"아래 내용으로 테스트 호텔 지도화가 되었다고 했고, sso를 이용하여 가입 페이지에...",it says that test hotel mapping has been done ...
2,제인이 방에 들어오면 분위기가 바뀝니다.,the atmosphere changes when jane enters the room.
3,나는 네가 그 채팅 앱을 사용하는 것이 싫어.,i dislike you using that chatting app.
4,창의적인 아이디어를 통해 지역과 함께 발전하는 회사입니다.,it's a company growing with the community thro...
...,...,...
399995,그렇군요. 그런데 그날 오후에 서울 떠나서 다른 날로 일정을 잡을 수가 없습니다.,i see. but i cant schedule another date since ...
399996,그건 나에게 온 편지를 받기 위해서예요.,it is getting the letter which was sent to me.
399997,내 생각에는 silicone volume 값을 측정하는 화면만 있으면 될 것 같아.,"in my opinion, all we need is a screen for mea..."
399998,당신은 나를 충분히 안심 시켜 줬어요.,you relieved me enough.


### 데이터 셋 전처리

결측치를 제거하고, 특수문자나 한자를 제거해주겠습니다.

In [5]:
train_df.isnull().sum()

kor    0
eng    0
dtype: int64

In [6]:
train_df = train_df.dropna()

In [7]:
import re

# 특수 문자 제거 및 소문자화
def preprocess_text(text: str) -> str:
    text = text.lower()
    # text = re.sub(r"[^ㄱ-힣a-zA-Z0-9!?.,\']", r" ", text)
    text = re.sub(r"[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z!?.,\' ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [7]:
example = "亞 최고의 영화...!! 단연코 BEST 보는 내내 눈물 ㅠㅠ"
preprocess_text(example)

'최고의 영화...!! 단연코 best 보는 내내 눈물 ㅠㅠ'

In [9]:
from tqdm.notebook import tqdm

tqdm.pandas()
train_df["kor"] = train_df["kor"].progress_apply(lambda x: preprocess_text(x))
train_df["eng"] = train_df["eng"].progress_apply(lambda x: preprocess_text(x))

  0%|          | 0/400000 [00:00<?, ?it/s]

  0%|          | 0/400000 [00:00<?, ?it/s]

In [10]:
train_df

,kor,eng
0,"mslee가 부재중이셔서, 당신에게 연락드립니다.","since ms.lee is absence, i contacted you."
1,"아래 내용으로 테스트 호텔 지도화가 되었다고 했고, sso를 이용하여 가입 페이지에...",it says that test hotel mapping has been done ...
2,제인이 방에 들어오면 분위기가 바뀝니다.,the atmosphere changes when jane enters the room.
3,나는 네가 그 채팅 앱을 사용하는 것이 싫어.,i dislike you using that chatting app.
4,창의적인 아이디어를 통해 지역과 함께 발전하는 회사입니다.,it's a company growing with the community thro...
...,...,...
399995,그렇군요. 그런데 그날 오후에 서울 떠나서 다른 날로 일정을 잡을 수가 없습니다.,i see. but i cant schedule another date since ...
399996,그건 나에게 온 편지를 받기 위해서예요.,it is getting the letter which was sent to me.
399997,내 생각에는 silicone volume 값을 측정하는 화면만 있으면 될 것 같아.,"in my opinion, all we need is a screen for mea..."
399998,당신은 나를 충분히 안심 시켜 줬어요.,you relieved me enough.


## huggingface tokenizers

huggingface는 AI 스타트업으로 오픈 소스 라이브러리로 유명합니다. 특히 NLP 분야에서는 huggingface에서 제공하는 트랜스포머 모델을 사용하는 것이 거의 표준으로 자리잡았습니다. 주요 라이브러리는 아래와 같습니다. 

- transformers: 트랜스포머 기본 모델과 이를 응용한 NLP 모델들을 제공
- tokenizers: subword tokenizer 제공

huggingface의 tokenizers는 subword tokenizer의 구현체입니다. 이를 사용하여 tokenizer를 학습시켜 보겠습니다.

In [10]:
!pip install tokenizers

DEPRECATION: pytorch-lightning 1.8.3.post1 has a non-standard dependency specifier torch>=1.9.*. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pytorch-lightning or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063
DEPRECATION: mecab-python 0.996-ko-0.9.2 has a non-standard version number. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of mecab-python or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063


### Trainer
WordPiece 기반의 subword tokenizer를 만들어보겠습니다.  먼저 tokenizer를 학습시키기 위한 trainer를 만들어줍니다. 이 때, special tokens를 넣어주어야 하는데, 각 토큰의 의미는 다음과 같습니다.

- [PAD]: padding의 약자로 문장 간에 길이를 맞춰주기 위해 일부러 채워넣은 토큰을 의미합니다.
- [UNK]: unknown의 약자로 인식하지 못한 토큰을 나타냅니다.
- [SOS]: start of sentence의 약자로 문장의 시작을 표시해줍니다.
- [EOS]: end of sentence의 약자로 문장의 끝을 표시해줍니다.

In [11]:
from tokenizers.trainers import WordPieceTrainer

trainer = WordPieceTrainer(
    vocab_size=10000,
    min_frequency=5,
    special_tokens=["[PAD]", "[UNK]", "[SOS]", "[EOS]"]
)

### Tokenizer

이제 tokenizer 객체를 만들어주고 trainer를 이용해서 학습시켜 줍니다.

In [17]:
train_df.iloc[:10]["kor"] + " " + train_df.iloc[:10]["eng"]

0    mslee가 부재중이셔서, 당신에게 연락드립니다. since ms.lee is ab...
1    아래 내용으로 테스트 호텔 지도화가 되었다고 했고, sso를 이용하여 가입 페이지에...
2    제인이 방에 들어오면 분위기가 바뀝니다. the atmosphere changes ...
3    나는 네가 그 채팅 앱을 사용하는 것이 싫어. i dislike you using ...
4    창의적인 아이디어를 통해 지역과 함께 발전하는 회사입니다. it's a compan...
5    제 엔진이 죽어서 시동이 안 걸려요. my motor died, and so it ...
6    가스회사 전화번호 좀 알려주세요. tell me the phone number of...
7    당신 물건은 예정된 스케줄대로 선적될 것입니다. your products will ...
8    주디가 휴가 가서 내가 잠시 이 업무를 맡았어. i'm handling this w...
9    스티커는 겉면 플라스틱 봉지에는 필요 없습니다. no sticker is neede...
dtype: object

In [18]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece 
from tokenizers.pre_tokenizers import Whitespace

def batch_iterator(df, batch_size=1000):
    for i in range(0, len(df), batch_size):
        batch_df = df.iloc[i:i+batch_size]
        yield batch_df["kor"] + " " + batch_df["eng"]

In [19]:
tokenizer = Tokenizer(WordPiece())
tokenizer.pre_tokenizer = Whitespace()
tokenizer.train_from_iterator(batch_iterator(train_df),  trainer=trainer)

## tokenizer 확인

tokenizer가 잘 학습되었는지 확인해보겠습니다.

### vocab 확인

먼저 tokenizer vocab에 어떤 토큰들이 추가되었는지 살펴보겠습니다.

In [20]:
vocabs = tokenizer.get_vocab()
sorted_vocabs = sorted(vocabs.items(), key=lambda x: x[1])

In [21]:
print(sorted_vocabs)

[('[PAD]', 0), ('[UNK]', 1), ('[SOS]', 2), ('[EOS]', 3), ('!', 4), ("'", 5), (',', 6), ('.', 7), ('?', 8), ('a', 9), ('b', 10), ('c', 11), ('d', 12), ('e', 13), ('f', 14), ('g', 15), ('h', 16), ('i', 17), ('j', 18), ('k', 19), ('l', 20), ('m', 21), ('n', 22), ('o', 23), ('p', 24), ('q', 25), ('r', 26), ('s', 27), ('t', 28), ('u', 29), ('v', 30), ('w', 31), ('x', 32), ('y', 33), ('z', 34), ('ㄷ', 35), ('ㄹ', 36), ('ㅅ', 37), ('ㅆ', 38), ('ㅇ', 39), ('ㅈ', 40), ('ㅏ', 41), ('ㅔ', 42), ('ㅠ', 43), ('ㅡ', 44), ('가', 45), ('각', 46), ('간', 47), ('갇', 48), ('갈', 49), ('갉', 50), ('감', 51), ('갑', 52), ('값', 53), ('갓', 54), ('갔', 55), ('강', 56), ('갖', 57), ('갗', 58), ('같', 59), ('갚', 60), ('갛', 61), ('개', 62), ('객', 63), ('갠', 64), ('갤', 65), ('갭', 66), ('갯', 67), ('갰', 68), ('갱', 69), ('갸', 70), ('걀', 71), ('걔', 72), ('거', 73), ('걱', 74), ('건', 75), ('걷', 76), ('걸', 77), ('검', 78), ('겁', 79), ('것', 80), ('겄', 81), ('겆', 82), ('겉', 83), ('겊', 84), ('게', 85), ('겐', 86), ('겔', 87), ('겜', 88), ('겟', 89), ('겠

4000 ~ 5000개까지는 초기 기본 토큰들로 채워져 있고, 그 뒤로는 함께 자주 등장하는 글자끼리 묶인 토큰들을 확인할 수 있습니다. 앞에 ##이 붙은 토큰들은 단어의 시작 지점이 아닌 위치에 등장하는 토큰들입니다.

### 샘플 토큰화

예시 문장들을 토큰화 해보겠습니다.

In [16]:
samples = [
    "너무 재밌어요, 꿀잼 인정!",
    "보다가 중간에 졸았습니다 ㅠㅠ 비추에요"
]

In [17]:
for sample in samples:
    output = tokenizer.encode(sample)
    print(output.ids)
    print(output.tokens)

[4882, 5396, 6, 6020, 6309, 4]
['너무', '재밌어요', ',', '꿀잼', '인정', '!']
[5285, 5836, 1843, 6944, 5035, 6208, 5377]
['보다가', '중간에', '졸', '##았습니다', 'ㅠㅠ', '비추', '##에요']


### 저장

잘 학습된 것을 확인했다면 파일에 저장하겠습니다.

In [22]:
tokenizer.save("./data/translation_tokenizer.json")

## 정리

이번 챕터에서는 huggingface에서 제공하는 tokenizers 라이브러리를 이용해서 직접 subword tokenizer를 만들어보았습니다. subword tokenizer는 활용도가 매우 높으니, 사용법을 잘 기억해주시기 바랍니다.